In [ ]:
pip install scikit-image


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir ~/.kaggle

In [ ]:
!mkdir ~/.kaggle

!cp kaggle.json ~/.kaggle/

In [ ]:
! pip install -q kaggle

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import shutil

# Specify the directory path
dir_path = '/new_image/content/breast-histopathology-images/new_data/'

# Check if the directory exists
if os.path.exists(dir_path):
    # Delete the directory and its contents
    shutil.rmtree(dir_path)
    print(f"{dir_path} and its contents have been deleted.")
else:
    print(f"The directory {dir_path} does not exist.")


In [ ]:
!kaggle datasets download -d paultimothymooney/breast-histopathology-images

In [ ]:
!unzip breast-histopathology-images.zip -d breast-histopathology-images

In [ ]:
!unzip '/content/drive/MyDrive/Colab Notebooks/newdata_image.zip' -d '/new_image'

In [ ]:

import shutil

# Specify the source and destination paths
source_path = '/new_image/content/breast-histopathology-images/new_data'
destination_path = '/content/breast-histopathology-images/IDC_regular_ps50_idx5/new_image'

# Move the folder
shutil.move(source_path, destination_path)

print(f"Moved folder from {source_path} to {destination_path}")

In [ ]:
import shutil

# Specify the source and destination paths
source_path = '/content/drive/MyDrive/phase2_model.pt'
destination_path = '/content/'

# Move the folder
shutil.move(source_path, destination_path)

print(f"Moved folder from {source_path} to {destination_path}")

In [ ]:
import shutil

# Specify the source and destination paths
source_path = '/content/breast-histopathology-images/IDC_regular_ps50_idx5'
destination_path = '/content/IDC'

# Move the folder
shutil.move(source_path, destination_path)

print(f"Moved folder from {source_path} to {destination_path}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from skimage.io import imread
import cv2


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
from PIL import Image


import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR, CyclicLR
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight


from glob import glob
from skimage.io import imread
from os import listdir

import time
import copy
from tqdm.notebook import tqdm
import os

In [ ]:
import glob  # Ensure glob is correctly imported

images1 = glob.glob('/content/breast-histopathology-images/**/*.png', recursive=True)
images2 = glob.glob('/content/IDC/new_image/new_data/*.png', recursive=True)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 5))
for i in range(5):
    img = cv2.imread(images1[i])
    axes[i].imshow(img)

In [ ]:
df1 = pd.DataFrame(index=np.arange(0, len(images1)), columns=["path", "target"])
df2 = pd.DataFrame(index=np.arange(0, len(images2)), columns=["path", "target"])
for i in range(len(images1)):
    df1.loc[i, "path"] = images1[i]
    df1.loc[i, "target"] = int(images1[i][-5])
for i in range(len(images2)):
    df2.loc[i, "path"] = images2[i]
    df2.loc[i, "target"] = 1

In [ ]:
from sklearn.utils import shuffle
# Concatenate the DataFrames
df_combined = pd.concat([df1, df2], ignore_index=True)
df = shuffle(df_combined, random_state=42)
df

In [ ]:
pos = df[df.target==1]
neg = df[df.target==0]

In [ ]:
df['target'].value_counts()


In [ ]:
print(pos.shape)
print(neg.shape)

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(20,10))
fig.suptitle('Cancer', fontsize=20)
for i in range(4):
    for j in range(10):
        image = imread(pos.iloc[j + 10*i]["path"])
        axes[i,j].imshow(image)
plt.tight_layout(rect=[0, 0, 1, 0.96])

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(20,10))
fig.suptitle('No Cancer', fontsize=20)
for i in range(4):
    for j in range(10):
        image = imread(neg.iloc[j + 10*i]["path"])
        axes[i,j].imshow(image)
plt.tight_layout(rect=[0, 0, 1, 0.96])

In [ ]:

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=df['target'].value_counts().index, y = df['target'].value_counts(), palette='Set1')
plt.xlabel('State of Cancer')
plt.ylabel('Number of Patients')
plt.title('Distribution of Cancer States')
plt.legend(handles=ax.patches[:2], labels=['0 = No Cancer', '1 = Cancer'], loc='upper right')
plt.show()

In [ ]:
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import os
import random



# Create a PyTorch Dataset
from torch.utils.data import Dataset

class ImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):

        return len(self.dataframe)



    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = row['target']

        if self.transform:
            image = self.transform(image)

        return image, label




In [ ]:
df.to_csv('/content/my_dataframe.csv', index=False)

In [ ]:
!zip -r /content/newdata_image.zip /content/breast-histopathology-images/new_data

In [ ]:
# Define dataset and DataLoader
transform = transforms.Compose([transforms.ToTensor()])
dataset = ImageDataset(df, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:

df["target"] = df["target"].astype(int)
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=df['target'].value_counts().index, y = df['target'].value_counts(), palette='Set1')
plt.xlabel('State of Cancer')
plt.ylabel('Number of Patients')
plt.title('Distribution of Cancer States')
plt.legend(handles=ax.patches[:2], labels=['0 = No Cancer', '1 = Cancer'], loc='upper right')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# Train-test split
train_df, test_df1 = train_test_split(df, test_size=0.3, stratify=df['target'], random_state=42)
test_df , val_df =  train_test_split(test_df1, test_size=0.5, stratify=test_df1['target'], random_state=42)

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((50, 50)),  # Resize all images to 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)

])

test_transform = transforms.Compose([
    transforms.Resize((50, 50)),  # Resize all images to 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)

])

val_transform = transforms.Compose([
    transforms.Resize((50, 50)),  # Resize all images to 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)

])
# Create separate PyTorch Datasets for training and testing
train_set = ImageDataset(train_df, transform=train_transform)
test_set = ImageDataset(test_df, transform=test_transform)
val_set = ImageDataset(val_df, transform=val_transform)
# Create DataLoaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
test_df.shape


In [ ]:
def normalize_image(image):
  image_min = image.min()
  image_max = image.max()
  image.clamp_(min = image_min, max = image_max)
  image.add_(-image_min).div_(image_max - image_min + 1e-5)
  return image

def plot_images(images, labels, classes, normalize=True):
  n_images = len(images)

  rows = int(np.sqrt(n_images))
  cols = int(np.sqrt(n_images))

  fig = plt.figure(figsize=(10, 10))

  for i in range(rows*cols):

    ax = fig.add_subplot(rows, cols, i+1)

    image = images[i]
    if normalize:
      image = normalize_image(image)

    ax.imshow(image.permute(1, 2, 0).cpu().numpy())
    ax.set_title(classes[labels[i]])
    ax.axis('off')

In [ ]:
# Define a Residual Block using Sequential
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.main_path(x)
        out += self.shortcut(x)
        return nn.ReLU()(out)

In [ ]:
def conv3x3_bn_af(in_channels, out_channels, use_residual=False, dropout_rate=0.1):
    if use_residual:
        return ResidualBlock(in_channels, out_channels)
    else:
        layers = [
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        ]
        if dropout_rate > 0.0:
            layers.append(nn.Dropout2d(dropout_rate))
        return nn.Sequential(*layers)


# Define the CNN model with optional Residual Connections
class CNN(nn.Module):
    def __init__(self, use_residual=False):
        super(CNN, self).__init__()
        self.network = nn.Sequential(
            conv3x3_bn_af(3, 64, use_residual=use_residual),
            conv3x3_bn_af(64, 64, use_residual=use_residual),
            nn.MaxPool2d(2, 2),  # 16x16

            conv3x3_bn_af(64, 128, use_residual=use_residual),
            conv3x3_bn_af(128, 128, use_residual=use_residual),
            nn.MaxPool2d(2, 2),  # 8x8

            conv3x3_bn_af(128, 256, use_residual=use_residual),
            conv3x3_bn_af(256, 256, use_residual=use_residual),
            nn.MaxPool2d(2, 2),  # 4x4

            conv3x3_bn_af(256, 512, use_residual=use_residual),
            conv3x3_bn_af(512, 512, use_residual=use_residual),
            nn.AdaptiveAvgPool2d(output_size=(1, 1)),  # 1x1

            nn.Flatten(),
            nn.Linear(512, 2)  # classifier
        )

    def forward(self, x):
        return self.network(x)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
pip install torchmetrics

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics import Precision, Recall, F1Score, AUROC

# Initialize metrics for train/validation and test
precision_metric = Precision(task="multiclass", num_classes=2).to(device)
recall_metric = Recall(task="multiclass", num_classes=2).to(device)
f1_metric = F1Score(task="multiclass", num_classes=2).to(device)
auc_metric = AUROC(task="multiclass", num_classes=2).to(device)

In [ ]:

# Plotting function
def plot_metrics(loss_train_hist, loss_valid_hist, acc_train_hist, acc_valid_hist,
                 precision_train_hist, precision_valid_hist,
                 recall_train_hist, recall_valid_hist):
    epochs = range(1, len(loss_train_hist) + 1)

    # Loss plot
    plt.figure(figsize=(12, 8))
    plt.subplot(2, 2, 1)
    plt.plot(epochs, loss_train_hist, label="Train Loss")
    plt.plot(epochs, loss_valid_hist, label="Validation Loss")
    plt.title("Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    # Accuracy plot
    plt.subplot(2, 2, 2)
    plt.plot(epochs, acc_train_hist, label="Train Accuracy")
    plt.plot(epochs, acc_valid_hist, label="Validation Accuracy")
    plt.title("Accuracy over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Precision plot
    plt.subplot(2, 2, 3)
    plt.plot(epochs, precision_train_hist, label="Train Precision")
    plt.plot(epochs, precision_valid_hist, label="Validation Precision")
    plt.title("Precision over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Precision")
    plt.legend()

    # Recall plot
    plt.subplot(2, 2, 4)
    plt.plot(epochs, recall_train_hist, label="Train Recall")
    plt.plot(epochs, recall_valid_hist, label="Validation Recall")
    plt.title("Recall over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Recall")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
# Update train_one_epoch function to track precision/recall
def train_one_epoch(model, train_loader, loss_fn, optimizer, epoch=None):
    model.train()
    loss_train = AverageMeter()
    acc_train = Accuracy(task="multiclass", num_classes=2).to(device)
    precision_train = Precision(task="multiclass", num_classes=2).to(device)
    recall_train = Recall(task="multiclass", num_classes=2).to(device)

    with tqdm(train_loader, unit="batch") as tepoch:
        for inputs, targets in tepoch:
            if epoch is not None:
                tepoch.set_description(f"Epoch {epoch}")
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)

            loss = loss_fn(outputs, targets)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Update metrics
            loss_train.update(loss.item())
            acc_train(outputs, targets.int())
            precision_train(outputs, targets.int())
            recall_train(outputs, targets.int())

            tepoch.set_postfix(loss=loss_train.avg,
                               accuracy=100. * acc_train.compute().item(),
                               precision=100. * precision_train.compute().item(),
                               recall=100. * recall_train.compute().item())

    return model, loss_train.avg, acc_train.compute().item(), precision_train.compute().item(), recall_train.compute().item()

In [ ]:
# Update validation function to include precision/recall
def validation(model, test_loader, loss_fn):
    model.eval()
    with torch.no_grad():
        loss_valid = AverageMeter()
        acc_valid = Accuracy(task="multiclass", num_classes=2).to(device)
        precision_valid = Precision(task="multiclass", num_classes=2).to(device)
        recall_valid = Recall(task="multiclass", num_classes=2).to(device)

        for i, (inputs, targets) in enumerate(test_loader):
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            # Update metrics
            loss_valid.update(loss.item())
            acc_valid(outputs, targets.int())
            precision_valid(outputs, targets.int())
            recall_valid(outputs, targets.int())

    return loss_valid.avg, acc_valid.compute().item(), precision_valid.compute().item(), recall_valid.compute().item()

In [ ]:
from torchmetrics import Precision, Recall , Accuracy

In [ ]:
model = CNN(use_residual = True).to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

In [ ]:
loss_train_hist = []
loss_valid_hist = []

acc_train_hist = []
acc_valid_hist = []
# Training Loop Updates
precision_train_hist = []
precision_valid_hist = []
recall_train_hist = []
recall_valid_hist = []

best_loss_valid = torch.inf
epoch_counter = 0

In [ ]:
# Ensure cuDNN is enabled
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True  # For consistent input sizes

In [ ]:
num_epochs = 10

def train():
    global model , epoch_counter
    checkpoint_path = '/content/model (16).pt'
    best_loss_valid = float('inf')

    # Load checkpoint if it exists
    try:
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        epoch_counter = checkpoint['step'] + 1
        print(f"Resumed training from epoch {epoch_counter}")
    except FileNotFoundError:
        print("No checkpoint found. Starting training from scratch.")
        epoch_counter = 0


    model.train()
    for epoch in range(num_epochs):
        model, loss_train, acc_train, precision_train, recall_train = train_one_epoch(
            model, train_loader, loss_fn, optimizer, epoch
        )

        loss_valid, acc_valid, precision_valid, recall_valid = validation(
            model, val_loader, loss_fn
        )

        loss_train_hist.append(loss_train)
        loss_valid_hist.append(loss_valid)
        acc_train_hist.append(acc_train)
        acc_valid_hist.append(acc_valid)
        precision_train_hist.append(precision_train)
        precision_valid_hist.append(precision_valid)
        recall_train_hist.append(recall_train)
        recall_valid_hist.append(recall_valid)

        # Save the best model
        if loss_valid < best_loss_valid:
            torch.save({
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'step': epoch
            }, checkpoint_path)
            best_loss_valid = loss_valid
            print(f"Saved best model at epoch {epoch} with val_loss: {loss_valid:.4f}")

        print(f'Valid: Loss = {loss_valid:.4f}, Acc = {acc_valid:.4f}')
        print()
        epoch_counter+=1
train()

In [ ]:
plot_metrics(loss_train_hist, loss_valid_hist, acc_train_hist, acc_valid_hist, precision_train_hist, precision_valid_hist, recall_train_hist,recall_valid_hist
)

In [ ]:
epochs = list(range(1, len(loss_train_hist) + 1))
# Create subplots for better visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss plot
axes[0, 0].plot(epochs, loss_train_hist, label='Train Loss', marker='o')
axes[0, 0].plot(epochs, loss_valid_hist, label='Validation Loss', marker='o')
axes[0, 0].set_title('Loss')
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Accuracy plot
axes[0, 1].plot(epochs, acc_train_hist, label='Train Accuracy', marker='o')
axes[0, 1].plot(epochs, acc_valid_hist, label='Validation Accuracy', marker='o')
axes[0, 1].set_title('Accuracy')
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision plot
axes[1, 0].plot(epochs, precision_train_hist, label='Train Precision', marker='o')
axes[1, 0].plot(epochs, precision_valid_hist, label='Validation Precision', marker='o')
axes[1, 0].set_title('Precision')
axes[1, 0].set_xlabel('Epochs')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Recall plot
axes[1, 1].plot(epochs, recall_train_hist, label='Train Recall', marker='o')
axes[1, 1].plot(epochs, recall_valid_hist, label='Validation Recall', marker='o')
axes[1, 1].set_title('Recall')
axes[1, 1].set_xlabel('Epochs')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True)

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
from sklearn.utils import shuffle
# Concatenate the DataFrames
df_test_new = test_df
df_test_new = shuffle(df_test_new, random_state=42)
print(df_test_new)

In [ ]:
# Drop rows from df1 that are also in df2
df_test_new = df_test_new[~df_test_new.isin(df2.to_dict(orient='list')).all(axis=1)]

print("Updated df1:")
print(df_test_new)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torchmetrics

In [ ]:
from torchmetrics import Precision, Recall , Accuracy,F1Score,  AUROC

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((50, 50)),  # Resize all images to 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)

])
set_test_new = ImageDataset(df_test_new, transform=test_transform)
test_loader = DataLoader(set_test_new, batch_size=32, shuffle=True)

In [ ]:
checkpoint = torch.load("/content/phase2_model.pt", map_location=torch.device('cpu'))
print(checkpoint.keys())
# Extract the state_dict from the checkpoint
state_dict = checkpoint['state_dict']

# Load the state_dict into the model
model = CNN(use_residual=True).to(torch.device('cpu'))
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()


In [ ]:
test_f1 = F1Score(task="multiclass", num_classes=2).to(device)
test_auc = AUROC(task="multiclass", num_classes=2).to(device)

model.eval()
with torch.no_grad():
    for inputs, targets in test_loader:
        # Move inputs and targets to the same device as the model
        inputs, targets = inputs.to(device), targets.to(device)

        # Perform forward pass
        outputs = model(inputs)

        test_f1(outputs, targets.int())
        test_auc(outputs, targets.int())

In [ ]:

print(f"Test F1 Score: {test_f1.compute().item():.4f}")
print(f"Test AUC: {test_auc.compute().item():.4f}")


In [ ]:
import torch.nn.functional as F

# Grad-CAM implementation
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        self.target_layer.register_forward_hook(self.save_activations)
        self.target_layer.register_backward_hook(self.save_gradients)

    def save_activations(self, module, input, output):
        self.activations = output

    def save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_tensor, target_class):
        # Forward pass
        output = self.model(input_tensor)
        pred_class = output.argmax(dim=1).item()

        # Backward pass for target class
        self.model.zero_grad()
        target = output[0, target_class]
        target.backward()

        # Compute weights
        gradients = self.gradients.cpu().data.numpy()
        activations = self.activations.cpu().data.numpy()
        weights = np.mean(gradients, axis=(2, 3))

        # Compute Grad-CAM
        cam = np.zeros(activations.shape[2:], dtype=np.float32)
        for i, w in enumerate(weights[0]):
            cam += w * activations[0, i, :, :]

        cam = np.maximum(cam, 0)
        cam = cam / cam.max()  # Normalize
        return cam




In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Define the transformation to apply to the image
transform = transforms.Compose([
    transforms.Resize((50, 50)),  # Resize images to 50x50
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize for pre-trained models
])

# Load model and checkpoint
model = CNN(use_residual=True).to(device)
checkpoint = torch.load('/content/model (16).pt')

if 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])

model.eval()

# Load example data
positive_sample = Image.open('/content/breast-histopathology-images/9345/1/9345_idx5_x2001_y151_class1.png')
negative_sample = Image.open('/content/breast-histopathology-images/12890/0/12890_idx5_x801_y1451_class0.png')

# Apply the transformation to the images
positive_tensor = transform(positive_sample).unsqueeze(0).to(device)  # Add batch dimension
negative_tensor = transform(negative_sample).unsqueeze(0).to(device)  # Add batch dimension

# Check the shape of the tensors
print("Positive tensor shape:", positive_tensor.shape)
print("Negative tensor shape:", negative_tensor.shape)

# Select target layer
target_layer = model.network[-4]  # Ensure this is the correct layer for Grad-CAM

# Initialize Grad-CAM
grad_cam = GradCAM(model, target_layer)

# Generate Heatmaps
positive_cam = grad_cam.generate(positive_tensor, target_class=1)  # Should return a 2D array
negative_cam = grad_cam.generate(negative_tensor, target_class=0)

# Visualize Heatmaps
def show_heatmap(image_tensor, cam, title):
    # Convert image tensor to numpy array
    image = image_tensor.squeeze(0).cpu().permute(1, 2, 0).numpy()
    image = (image - image.min()) / (image.max() - image.min())  # Normalize image to 0-1

    # Resize the CAM to match the image dimensions
    cam = cv2.resize(cam, (image.shape[1], image.shape[0]))

    # Normalize the CAM to 0-255 range
    cam = np.uint8(255 * cam / cam.max())
    cam_colored = cv2.applyColorMap(cam, cv2.COLORMAP_JET)  # Change COLORMAP_HOT to your preferred option


    # Overlay the heatmap on the image
    heatmap_overlay = cv2.addWeighted((image * 255).astype(np.uint8), 0.5, cam_colored, 0.5, 0)

    # Display the image with the heatmap
    plt.figure(figsize=(6, 6))
    plt.imshow(heatmap_overlay)
    plt.title(title)
    plt.axis('off')
    plt.show()

# Display results
show_heatmap(positive_tensor, positive_cam, "Positive Sample (Cancerous)")
show_heatmap(negative_tensor, negative_cam, "Negative Sample (Non-Cancerous)")
